In [12]:
import os
import numpy as np
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

SEED = 42
DATASET_PATH = "../DATASET"
OUTPUT_PATH  = DATASET_PATH

print('Imports OK')

Imports OK


In [13]:
csv_files = [
    os.path.join(DATASET_PATH, f)
    for f in os.listdir(DATASET_PATH)
    if f.endswith('.csv') and 'resampled' not in f.lower()  # skip already-saved outputs
]

chunks = []
for file in csv_files:
    print('Loading:', os.path.basename(file))
    for chunk in pd.read_csv(file, chunksize=200_000):
        chunk.columns = chunk.columns.str.strip()  # strip here, once
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f'\nFull dataset shape: {df.shape}')
print(df[' Label'].value_counts() if ' Label' in df.columns else df['Label'].value_counts())

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv

Full dataset shape: (2830743, 79)
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection  

In [14]:
df = df.drop_duplicates()
df = df.rename(columns={' Label': 'Label'})

LABEL_MAP = {
    # DDoS / DoS family   1
    'DDoS': 1,
    'DoS Hulk': 1,
    'DoS GoldenEye': 1,
    'DoS Slowhttptest': 1,
    'DoS slowloris': 1,
    # PortScan    2
    'PortScan': 2,
    # BruteForce family    3
    'FTP-Patator': 3,
    'SSH-Patator': 3,
    'Web Attack \u2013 Brute Force': 3,
    # Benign   0
    'BENIGN': 0,
    # Everything else    4 (General)
}

df['Label'] = df['Label'].map(lambda x: LABEL_MAP.get(x, 4))

print('Label distribution after mapping:')
print(df['Label'].value_counts())

Label distribution after mapping:
Label
0    2096484
1     321764
2      90819
3       9152
4       4143
Name: count, dtype: int64


In [15]:
df_anomaly = df[df['Label'] != 0].copy()



In [16]:
from sklearn.model_selection import train_test_split
X = df_anomaly.drop(columns=['Label'])
y = df_anomaly['Label'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=SEED, stratify=y)
X_train_final,X_val,y_train_final,y_val=train_test_split(X_train, y_train, test_size=0.1, random_state=SEED, stratify=y_train)

## 4. Clean — remove inf / NaN (before any split)

In [17]:
import numpy as np

def clean(X, train_median=None, clip_val=1e6):
    X = X.copy()

    # Replace inf with NaN
    X.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Fit median ONLY on training data
    if train_median is None:
        train_median = X.median(numeric_only=True)

    # Fill missing using TRAIN median
    X = X.fillna(train_median)

    # Clip extreme values
    X = X.clip(-clip_val, clip_val)

    return X, train_median

In [18]:
# ✅ Fit ONLY on training
X_train_clean, train_median = clean(X_train_final)

# ✅ Apply same median to val & test
X_val_clean, _ = clean(X_val, train_median)
X_test_clean, _ = clean(X_test, train_median)

In [19]:


# ── Step A: Undersample majority classes ────────────────────────────────────
TARGET_PER_CLASS = 300_000  # adjust based on your RAM

counts = y_train_final.value_counts()
under_strategy = {
    cls: min(TARGET_PER_CLASS, count)
    for cls, count in counts.items()
}

rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=SEED)
X_under, y_under = rus.fit_resample(X_train_clean, y_train_final)

print('After undersampling:')
print(pd.Series(y_under).value_counts())

After undersampling:
Label
1    260628
2     73564
3      7413
4      3356
Name: count, dtype: int64


In [20]:

counts_under = pd.Series(y_under).value_counts()
imbalance = counts_under.max() / counts_under.min()
print(f'Imbalance ratio before SMOTE: {imbalance:.1f}x')

if imbalance > 2:
    smote = SMOTE(random_state=SEED, k_neighbors=5)
    X_res, y_res = smote.fit_resample(X_under, y_under)
    print('After SMOTE:')
    print(pd.Series(y_res).value_counts())
else:
    X_res, y_res = X_under, y_under
    print('Imbalance acceptable — skipping SMOTE')

Imbalance ratio before SMOTE: 77.7x
After SMOTE:
Label
1    260628
2    260628
3    260628
4    260628
Name: count, dtype: int64


In [21]:
train_set = set(map(tuple, X_train_clean.values))
val_set   = set(map(tuple, X_val_clean.values))

overlap = train_set & val_set
mask = ~X_val_clean.apply(tuple, axis=1).isin(overlap)
X_val_clean = X_val_clean[mask]
y_val = y_val[mask]

In [22]:
train_set = set(map(tuple, X_train_clean.values))
val_set   = set(map(tuple, X_val_clean.values))

overlap = train_set & val_set
print("Overlap rows:", len(overlap))

Overlap rows: 0


In [23]:
import os
import pandas as pd

# Ensure feature columns are consistent
feature_cols = X_train_clean.columns

# ── Save TRAIN (SMOTE-resampled) ────────────────────────────────────────────
df_train = pd.DataFrame(X_res, columns=feature_cols)
df_train['Label'] = y_res

train_path = os.path.join(OUTPUT_PATH, 'stage2_train_resampled.csv')
df_train.to_csv(train_path, index=False)


# ── Save VALIDATION (clean, NOT resampled) ──────────────────────────────────
df_val = pd.DataFrame(X_val_clean.values, columns=feature_cols)
df_val['Label'] = y_val.values

val_path = os.path.join(OUTPUT_PATH, 'stage2_val_clean.csv')
df_val.to_csv(val_path, index=False)


# ── Save TEST (clean, NOT resampled) ────────────────────────────────────────
df_test = pd.DataFrame(X_test_clean.values, columns=feature_cols)
df_test['Label'] = y_test.values

test_path = os.path.join(OUTPUT_PATH, 'stage2_test_clean.csv')
df_test.to_csv(test_path, index=False)


# ── Logs ────────────────────────────────────────────────────────────────────
print(f'Saved TRAIN → {train_path}')
print(f'Saved VAL   → {val_path}')
print(f'Saved TEST  → {test_path}')

print(f'\nTrain shape : {df_train.shape}')
print(f'Val   shape : {df_val.shape}')
print(f'Test  shape : {df_test.shape}')

print(f'\nTrain distribution:\n{df_train["Label"].value_counts()}')
print(f'\nVal   distribution:\n{df_val["Label"].value_counts()}')
print(f'\nTest  distribution:\n{df_test["Label"].value_counts()}')

print(f'\nConfirm no benign rows in train: {(df_train["Label"] == 0).sum()}')
print(f'Confirm no benign rows in val  : {(df_val["Label"] == 0).sum()}')
print(f'Confirm no benign rows in test : {(df_test["Label"] == 0).sum()}')

Saved TRAIN → ../DATASET\stage2_train_resampled.csv
Saved VAL   → ../DATASET\stage2_val_clean.csv
Saved TEST  → ../DATASET\stage2_test_clean.csv

Train shape : (1042512, 79)
Val   shape : (38321, 79)
Test  shape : (42588, 79)

Train distribution:
Label
1    260628
2    260628
3    260628
4    260628
Name: count, dtype: int64

Val   distribution:
Label
1    28951
2     8173
3      824
4      373
Name: count, dtype: int64

Test  distribution:
Label
1    32177
2     9082
3      915
4      414
Name: count, dtype: int64

Confirm no benign rows in train: 0
Confirm no benign rows in val  : 0
Confirm no benign rows in test : 0
